In [0]:
%run /Workspace/Users/panchalraahul10@gmail.com/common/01_lakehouse_config

External lakehouse databases registered pointing to customer ADLS Gen2.


In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Gold Layer: Dimensional Modeling (Kimball Star Schema)
# MAGIC Reads cleaned Silver Delta paths, constructs conformed dimensions and fact tables,
# MAGIC and writes external Delta tables directly to ADLS Gen2 Gold paths.


# COMMAND ----------
import datetime
from decimal import Decimal
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

print("Connected to Lakehouse paths:")
print(f"Silver: {SILVER_BASE}")
print(f"Gold:   {GOLD_BASE}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## 1. Helper Functions for Path-Based Delta Persistence

# COMMAND ----------
def write_gold_delta(df, target_path, pk_cols=None, overwrite=False):
    """
    Writes or merges a DataFrame directly to an external ADLS Gen2 Delta path.
    Uses Delta ACID logs without requiring Hive Metastore or Unity Catalog DDL.
    """
    if overwrite or not DeltaTable.isDeltaTable(spark, target_path):
        print(f"Writing (overwrite) Delta table to: {target_path}")
        df.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(target_path)
    else:
        print(f"Merging into Delta table at: {target_path}")
        match_pred = " AND ".join([f"target.{col} = source.{col}" for col in pk_cols])
        delta_target = DeltaTable.forPath(spark, target_path)
        delta_target.alias("target").merge(
            df.alias("source"),
            match_pred
        ).whenMatchedUpdateAll() \
         .whenNotMatchedInsertAll() \
         .execute()
    print(f"✓ Saved: {target_path.split('/')[-1]}")

# COMMAND ----------
# MAGIC %md
# MAGIC ## 2. Conformed Dimensions (dim_date, dim_plan, dim_organization, dim_user)

# COMMAND ----------
# MAGIC %md
# MAGIC ### A. dim_date (Dynamic 10-Year Rolling Window)

# COMMAND ----------
# Generate dates starting from 2020 to current_year + 10
start_year = 2020
end_year = datetime.datetime.now().year + 10

# Create date sequence via Spark SQL
df_dates = spark.sql(f"""
    SELECT explode(sequence(to_date('{start_year}-01-01'), to_date('{end_year}-12-31'), interval 1 day)) as date_actual
""")

df_dim_date = df_dates \
    .withColumn("date_key", F.date_format("date_actual", "yyyyMMdd").cast("int")) \
    .withColumn("day_of_month", F.dayofmonth("date_actual")) \
    .withColumn("month", F.month("date_actual")) \
    .withColumn("month_name", F.date_format("date_actual", "MMMM")) \
    .withColumn("quarter", F.quarter("date_actual")) \
    .withColumn("year", F.year("date_actual")) \
    .withColumn("is_weekend", F.dayofweek("date_actual").isin([1, 7])) \
    .select("date_key", "date_actual", "day_of_month", "month", "month_name", "quarter", "year", "is_weekend")

# Append default -1 unknown date member for missing/null dates
df_unknown_date = spark.createDataFrame(
    [(-1, datetime.date(1970, 1, 1), 1, 1, "Unknown", 1, 1970, False)],
    schema=df_dim_date.schema
)

df_dim_date_final = df_dim_date.unionByName(df_unknown_date)

write_gold_delta(df_dim_date_final, f"{GOLD_BASE}/dim_date", overwrite=True)

# COMMAND ----------
# MAGIC %md
# MAGIC ### B. dim_plan (With Unknown Member -1)

# COMMAND ----------
df_cln_plans = spark.read.format("delta").load(f"{SILVER_BASE}/cln_plans")

# Generate surrogate key plan_key
win_plan = Window.orderBy("plan_id")
df_dim_plan_active = df_cln_plans \
    .withColumn("plan_key", F.row_number().over(win_plan).cast("int")) \
    .select("plan_key", "plan_id", "plan_name", "tier", "monthly_fee")

# Default -1 orphan member
df_unknown_plan = spark.createDataFrame(
    [(-1, -1, "Unknown Plan", "unknown", 0.00)],
    schema=df_dim_plan_active.schema
)

df_dim_plan = df_dim_plan_active.unionByName(df_unknown_plan)

write_gold_delta(df_dim_plan, f"{GOLD_BASE}/dim_plan", overwrite=True)

# COMMAND ----------
# MAGIC %md
# MAGIC ### C. dim_organization (With Unknown Member -1)

# COMMAND ----------
df_cln_orgs = spark.read.format("delta").load(f"{SILVER_BASE}/cln_organizations")

win_org = Window.orderBy("org_id")
df_dim_org_active = df_cln_orgs \
    .withColumn("org_key", F.row_number().over(win_org).cast("int")) \
    .withColumnRenamed("name", "org_name") \
    .select("org_key", "org_id", "org_name", "industry", "country_code", "is_active")

df_unknown_org = spark.createDataFrame(
    [(-1, -1, "Unknown Organization", "Unknown", "XX", False)],
    schema=df_dim_org_active.schema
)

df_dim_org = df_dim_org_active.unionByName(df_unknown_org)

write_gold_delta(df_dim_org, f"{GOLD_BASE}/dim_organization", overwrite=True)

# COMMAND ----------
# MAGIC %md
# MAGIC ### D. dim_user (Resolves org_key via dim_organization, With Unknown Member -1)

# COMMAND ----------
df_cln_users = spark.read.format("delta").load(f"{SILVER_BASE}/cln_users")

# Look up org_key from the saved dim_organization
dim_org_lookup = df_dim_org.select("org_id", "org_key")

win_user = Window.orderBy("user_id")
df_dim_user_active = df_cln_users.alias("u") \
    .join(dim_org_lookup.alias("o"), F.col("u.org_id") == F.col("o.org_id"), "left") \
    .withColumn("user_key", F.row_number().over(win_user).cast("int")) \
    .withColumn("org_key", F.coalesce(F.col("o.org_key"), F.lit(-1)).cast("int")) \
    .select("user_key", "user_id", "org_key", "email", "role", "is_active")

df_unknown_user = spark.createDataFrame(
    [(-1, -1, -1, "unknown@system.local", "Unknown", False)],
    schema=df_dim_user_active.schema
)

df_dim_user = df_dim_user_active.unionByName(df_unknown_user)

write_gold_delta(df_dim_user, f"{GOLD_BASE}/dim_user", overwrite=True)

# COMMAND ----------
# MAGIC %md
# MAGIC ## 3. Fact Tables (fact_billing, fact_subscriptions, fact_feature_usage, fact_logins)

# COMMAND ----------
# MAGIC %md
# MAGIC ### A. fact_billing

# COMMAND ----------
df_cln_invoices = spark.read.format("delta").load(f"{SILVER_BASE}/cln_invoices")
df_cln_subs = spark.read.format("delta").load(f"{SILVER_BASE}/cln_subscriptions")

# Join invoice -> subscription to get org_id and plan_id
df_inv_with_sub = df_cln_invoices.alias("i") \
    .join(df_cln_subs.alias("s"), F.col("i.subscription_id") == F.col("s.subscription_id"), "left") \
    .select(
        F.col("i.invoice_id"),
        F.col("i.billing_date"),
        F.col("i.amount_due"),
        F.col("i.amount_paid"),
        F.col("i.status"),
        F.col("i.is_anomaly"),
        F.col("s.org_id"),
        F.col("s.plan_id")
    )

# Map foreign keys to dimensions
win_billing = Window.orderBy("invoice_id")
df_fact_billing = df_inv_with_sub.alias("b") \
    .join(df_dim_org.select("org_id", "org_key").alias("o"), F.col("b.org_id") == F.col("o.org_id"), "left") \
    .join(df_dim_plan.select("plan_id", "plan_key").alias("p"), F.col("b.plan_id") == F.col("p.plan_id"), "left") \
    .withColumn("billing_key", F.row_number().over(win_billing).cast("bigint")) \
    .withColumn("date_key", F.coalesce(F.date_format("b.billing_date", "yyyyMMdd").cast("int"), F.lit(-1))) \
    .withColumn("org_key", F.coalesce(F.col("o.org_key"), F.lit(-1)).cast("int")) \
    .withColumn("plan_key", F.coalesce(F.col("p.plan_key"), F.lit(-1)).cast("int")) \
    .select(
        "billing_key",
        "invoice_id",
        "date_key",
        "org_key",
        "plan_key",
        "amount_due",
        "amount_paid",
        "status",
        "is_anomaly"
    )

write_gold_delta(df_fact_billing, f"{GOLD_BASE}/fact_billing", pk_cols=["invoice_id"])

# COMMAND ----------
# MAGIC %md
# MAGIC ### B. fact_subscriptions (MRR & Contract State)

# COMMAND ----------
win_sub = Window.orderBy("subscription_id")
df_fact_subscriptions = df_cln_subs.alias("s") \
    .join(df_dim_org.select("org_id", "org_key").alias("o"), F.col("s.org_id") == F.col("o.org_id"), "left") \
    .join(df_dim_plan.select("plan_id", "plan_key", "monthly_fee").alias("p"), F.col("s.plan_id") == F.col("p.plan_id"), "left") \
    .withColumn("subscription_fact_key", F.row_number().over(win_sub).cast("bigint")) \
    .withColumn("date_key", F.coalesce(F.date_format("s.started_at", "yyyyMMdd").cast("int"), F.lit(-1))) \
    .withColumn("org_key", F.coalesce(F.col("o.org_key"), F.lit(-1)).cast("int")) \
    .withColumn("plan_key", F.coalesce(F.col("p.plan_key"), F.lit(-1)).cast("int")) \
    .withColumn("is_active", F.col("s.status") == "active") \
    .withColumn(
        "mrr_amount",
        F.when(F.col("s.status") == "active", F.coalesce(F.col("p.monthly_fee"), F.lit(0.00))).otherwise(F.lit(0.00))
    ) \
    .select(
        "subscription_fact_key",
        "subscription_id",
        "date_key",
        "org_key",
        "plan_key",
        "status",
        "mrr_amount",
        "is_active"
    )

write_gold_delta(df_fact_subscriptions, f"{GOLD_BASE}/fact_subscriptions", pk_cols=["subscription_id"])

# COMMAND ----------
# MAGIC %md
# MAGIC ### C. fact_feature_usage

# COMMAND ----------
df_cln_usage = spark.read.format("delta").load(f"{SILVER_BASE}/cln_feature_usage")

win_usage = Window.orderBy(F.col("u.user_id"), F.col("u.org_id"), F.col("u.usage_date"), F.col("u.feature_name"))
df_fact_feature_usage = df_cln_usage.alias("u") \
    .join(df_dim_org.select("org_id", "org_key").alias("o"), F.col("u.org_id") == F.col("o.org_id"), "left") \
    .join(df_dim_user.select("user_id", "user_key").alias("usr"), F.col("u.user_id") == F.col("usr.user_id"), "left") \
    .withColumn("usage_fact_key", F.row_number().over(win_usage).cast("bigint")) \
    .withColumn("date_key", F.coalesce(F.date_format("u.usage_date", "yyyyMMdd").cast("int"), F.lit(-1))) \
    .withColumn("org_key", F.coalesce(F.col("o.org_key"), F.lit(-1)).cast("int")) \
    .withColumn("user_key", F.coalesce(F.col("usr.user_key"), F.lit(-1)).cast("int")) \
    .select(
        "usage_fact_key",
        "date_key",
        "org_key",
        "user_key",
        "feature_name",
        "usage_count"
    )

write_gold_delta(
    df_fact_feature_usage,
    f"{GOLD_BASE}/fact_feature_usage",
    pk_cols=["date_key", "org_key", "user_key", "feature_name"],
    overwrite=True
)

# COMMAND ----------
# MAGIC %md
# MAGIC ### D. fact_logins

# COMMAND ----------
df_cln_logins = spark.read.format("delta").load(f"{SILVER_BASE}/cln_logins")

win_login = Window.orderBy("event_id")
df_fact_logins = df_cln_logins.alias("l") \
    .join(df_dim_user.select("user_id", "user_key", "org_key").alias("u"), F.col("l.user_id") == F.col("u.user_id"), "left") \
    .withColumn("login_fact_key", F.row_number().over(win_login).cast("bigint")) \
    .withColumn("date_key", F.coalesce(F.date_format("l.event_timestamp", "yyyyMMdd").cast("int"), F.lit(-1))) \
    .withColumn("user_key", F.coalesce(F.col("u.user_key"), F.lit(-1)).cast("int")) \
    .withColumn("org_key", F.coalesce(F.col("u.org_key"), F.lit(-1)).cast("int")) \
    .select(
        "login_fact_key",
        "event_id",
        "date_key",
        "org_key",
        "user_key",
        F.col("l.event_timestamp").alias("login_timestamp"),
        F.col("l.login_success").alias("is_success"),
        "client_device"
    )

write_gold_delta(df_fact_logins, f"{GOLD_BASE}/fact_logins", pk_cols=["event_id"])

# COMMAND ----------
# MAGIC %md
# MAGIC ## 4. Gold Layer Verification

# COMMAND ----------
gold_entities = [
    "dim_date",
    "dim_plan",
    "dim_organization",
    "dim_user",
    "fact_billing",
    "fact_subscriptions",
    "fact_feature_usage",
    "fact_logins"
]

print("=== Gold Layer Table Row Counts ===")
for entity in gold_entities:
    path = f"{GOLD_BASE}/{entity}"
    cnt = spark.read.format("delta").load(path).count()
    print(f"• {entity:22} : {cnt:,} rows")

Connected to Lakehouse paths:
Silver: abfss://silver@datalakesaas.dfs.core.windows.net
Gold:   abfss://gold@datalakesaas.dfs.core.windows.net
Writing (overwrite) Delta table to: abfss://gold@datalakesaas.dfs.core.windows.net/dim_date
✓ Saved: dim_date


/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


Writing (overwrite) Delta table to: abfss://gold@datalakesaas.dfs.core.windows.net/dim_plan
✓ Saved: dim_plan
Writing (overwrite) Delta table to: abfss://gold@datalakesaas.dfs.core.windows.net/dim_organization
✓ Saved: dim_organization
Writing (overwrite) Delta table to: abfss://gold@datalakesaas.dfs.core.windows.net/dim_user
✓ Saved: dim_user
Merging into Delta table at: abfss://gold@datalakesaas.dfs.core.windows.net/fact_billing


/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✓ Saved: fact_billing
Merging into Delta table at: abfss://gold@datalakesaas.dfs.core.windows.net/fact_subscriptions


...

✓ Saved: fact_subscriptions
Writing (overwrite) Delta table to: abfss://gold@datalakesaas.dfs.core.windows.net/fact_feature_usage


/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✓ Saved: fact_feature_usage
Writing (overwrite) Delta table to: abfss://gold@datalakesaas.dfs.core.windows.net/fact_logins


/databricks/spark/python/pyspark/sql/connect/expressions.py:1134: UserWarning: WARN WindowExpression: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
  warnings.warn(


✓ Saved: fact_logins
=== Gold Layer Table Row Counts ===
• dim_date               : 6,211 rows
• dim_plan               : 6 rows
• dim_organization       : 301 rows
• dim_user               : 2,501 rows
• fact_billing           : 992 rows
• fact_subscriptions     : 344 rows
• fact_feature_usage     : 11,629 rows
• fact_logins            : 10,000 rows
